# Ingest Observability Data into DOG with OpenTelemetry

**Module 2: Ingesting Observability Data** · [Quiz](quiz02_ingesting_observability_data.ipynb) · [Course README](../../README.md)

Run this notebook from JupyterLab launched at the course root. The `lab.shell(...)` cells use the course root to mount the YAML configuration files. The Doris FE log path is discovered from the running `doris-obs` container.

## 1. Send logs to DOG

Use the following command to write log data to DOG through OTel. It reads the actual FE log directory from the `doris-obs` container's host mount, so it does not depend on the current working directory or the location where the DOG stack was checked out. Each collector joins the course's `doris-obs` Docker network and reaches Doris by its container name, `doris-obs` (see `endpoint` and `mysql_endpoint` in the YAML files):

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_CONTAINER="doris-obs"
CONFIG_DIR="level1/module02-data-ingest"
DORIS_FE_LOG_DIR="$(docker inspect --format '{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}' "$DORIS_CONTAINER")"

if [ -z "$DORIS_FE_LOG_DIR" ]; then
  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."
  echo "Run module 1's prepare_environment() cell first, then rerun this cell."
  exit 1
fi

# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.
DORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"

DORIS_FE_LOG="$DORIS_FE_LOG_DIR/fe.log"
if [ ! -f "$DORIS_FE_LOG" ]; then
  echo "The mounted Doris FE log was not found at $DORIS_FE_LOG"
  exit 1
fi

docker rm -f doris-obs-log2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-log2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/etc/otel_for_logs.yaml:ro" \
  -v "$DORIS_FE_LOG:/var/log/fe.log:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_logs.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-log2doris
""", title="Start the Doris log collector")


## 2. Send metrics to DOG

Use the following command to write metrics data to DOG through OTel. This example directly uses the metrics exposed by Doris:

In [ ]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-metric2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-metric2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_metrics.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-metric2doris
""", title="Start the Doris metric collector")


## 3. Send traces to DOG

Use the following command to start OTel. It will write trace data to DOG:

In [ ]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-traces2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-traces2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_traces.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-traces2doris
""", title="Start the trace collector")


## 4. Simulate trace data

Use the following Python code to simulate sending trace data. First, run the pip command:

In [ ]:
%pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http


Run the following command to send trace data to OTel. OTel will write the data to DOG:

In [ ]:
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

resource = Resource.create({"service.name": "python-demo-service"})

provider = TracerProvider(resource=resource)
# The DOG Collector's OTLP/HTTP receiver: http://127.0.0.1:24318 unless DORIS_OBS_OTLP_HTTP_PORT is set.
processor = BatchSpanProcessor(OTLPSpanExporter(endpoint=f"{lab.otlp_http_url}/v1/traces"))
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("demo-tracer")

with tracer.start_as_current_span("demo-span") as span:
    span.set_attribute("custom.attribute", "hello-otel")
    span.set_attribute("http.status_code", 200)
    print("Data sent successfully!")


## Next

Open [Quiz 2: Ingesting Observability Data](quiz02_ingesting_observability_data.ipynb), then continue to [Module 3: Visualizing Observability Data](../module03-data-visualization/lab03_data_visualization.ipynb).